In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

def get_fundamentals(ticker):
    stock = yf.Ticker(ticker)
    info  = stock.info
    
    # Price & valuation
    price       = info.get('currentPrice') or info.get('regularMarketPrice')
    pe          = info.get('trailingPE')
    fwd_pe      = info.get('forwardPE')
    pb          = info.get('priceToBook')
    peg         = info.get('pegRatio')
    ev_ebitda   = info.get('enterpriseToEbitda')
    div_yield   = info.get('dividendYield')
    
    # Quality
    roe         = info.get('returnOnEquity')
    roa         = info.get('returnOnAssets')
    gross_margin= info.get('grossMargins')
    op_margin   = info.get('operatingMargins')
    net_margin  = info.get('profitMargins')
    de_ratio    = info.get('debtToEquity')
    current_r   = info.get('currentRatio')
    fcf         = info.get('freeCashflow')
    
    # Growth
    rev_growth  = info.get('revenueGrowth')
    earn_growth = info.get('earningsGrowth')
    
    # Momentum
    week52_high = info.get('fiftyTwoWeekHigh')
    week52_low  = info.get('fiftyTwoWeekLow')
    ma50        = info.get('fiftyDayAverage')
    ma200       = info.get('twoHundredDayAverage')
    
    # Company info
    name        = info.get('longName', ticker)
    sector      = info.get('sector', 'N/A')
    industry    = info.get('industry', 'N/A')
    mktcap      = info.get('marketCap')
    beta        = info.get('beta')
    
    return {
        'name': name, 'sector': sector, 'industry': industry,
        'price': price, 'mktcap': mktcap, 'beta': beta,
        'pe': pe, 'fwd_pe': fwd_pe, 'pb': pb, 'peg': peg,
        'ev_ebitda': ev_ebitda, 'div_yield': div_yield,
        'roe': roe, 'roa': roa, 'gross_margin': gross_margin,
        'op_margin': op_margin, 'net_margin': net_margin,
        'de_ratio': de_ratio, 'current_r': current_r, 'fcf': fcf,
        'rev_growth': rev_growth, 'earn_growth': earn_growth,
        'week52_high': week52_high, 'week52_low': week52_low,
        'ma50': ma50, 'ma200': ma200,
    }

# Test it
ticker = 'MA'
data = get_fundamentals(ticker)
print(f"✅ Fetched data for {data['name']}")
print(f"   Sector: {data['sector']} | Industry: {data['industry']}")
print(f"   Price: ${data['price']} | Market Cap: ${data['mktcap']:,.0f}")

✅ Fetched data for Mastercard Incorporated
   Sector: Financial Services | Industry: Credit Services
   Price: $489.98 | Market Cap: $432,938,450,944


In [4]:
def generate_signal(data):
    scores = {}
    reasons = []

    # --- VALUATION SCORE (0-100, higher = cheaper) ---
    val = 0
    count = 0
    if data['pe']:
        if data['pe'] < 15:   v=100
        elif data['pe'] < 25: v=70
        elif data['pe'] < 35: v=40
        else:                  v=15
        val += v; count += 1
        reasons.append(f"P/E of {data['pe']:.1f} — {'attractive' if v>=70 else 'fair' if v>=40 else 'stretched'}")

    if data['fwd_pe']:
        if data['fwd_pe'] < 15:   v=100
        elif data['fwd_pe'] < 25: v=70
        elif data['fwd_pe'] < 35: v=40
        else:                      v=15
        val += v; count += 1

    if data['peg']:
        if data['peg'] < 1:   v=100
        elif data['peg'] < 2: v=65
        elif data['peg'] < 3: v=35
        else:                  v=10
        val += v; count += 1
        reasons.append(f"PEG of {data['peg']:.1f} — {'growth at reasonable price' if v>=65 else 'expensive relative to growth'}")

    if data['ev_ebitda']:
        if data['ev_ebitda'] < 10:  v=100
        elif data['ev_ebitda'] < 20: v=65
        elif data['ev_ebitda'] < 30: v=35
        else:                         v=10
        val += v; count += 1

    scores['valuation'] = val / count if count else 50

    # --- QUALITY SCORE (0-100) ---
    qual = 0
    count = 0
    if data['roe']:
        if data['roe'] > 0.30:   v=100
        elif data['roe'] > 0.15: v=70
        elif data['roe'] > 0.08: v=40
        else:                     v=15
        qual += v; count += 1
        reasons.append(f"ROE of {data['roe']*100:.1f}% — {'exceptional' if v>=100 else 'solid' if v>=70 else 'weak'}")

    if data['net_margin']:
        if data['net_margin'] > 0.25:  v=100
        elif data['net_margin'] > 0.15: v=75
        elif data['net_margin'] > 0.08: v=45
        else:                            v=15
        qual += v; count += 1
        reasons.append(f"Net margin of {data['net_margin']*100:.1f}% — {'best-in-class' if v>=100 else 'healthy' if v>=75 else 'thin'}")

    if data['de_ratio']:
        if data['de_ratio'] < 50:    v=100
        elif data['de_ratio'] < 100: v=70
        elif data['de_ratio'] < 200: v=40
        else:                         v=15
        qual += v; count += 1
        reasons.append(f"D/E ratio of {data['de_ratio']:.0f} — {'conservative balance sheet' if v>=70 else 'leveraged'}")

    if data['current_r']:
        if data['current_r'] > 2:   v=100
        elif data['current_r'] > 1: v=65
        else:                        v=20
        qual += v; count += 1

    scores['quality'] = qual / count if count else 50

    # --- GROWTH SCORE (0-100) ---
    grow = 0
    count = 0
    if data['rev_growth']:
        if data['rev_growth'] > 0.20:   v=100
        elif data['rev_growth'] > 0.10: v=75
        elif data['rev_growth'] > 0.05: v=50
        elif data['rev_growth'] > 0:    v=30
        else:                            v=10
        grow += v; count += 1
        reasons.append(f"Revenue growth of {data['rev_growth']*100:.1f}% — {'strong' if v>=75 else 'moderate' if v>=50 else 'slowing'}")

    if data['earn_growth']:
        if data['earn_growth'] > 0.20:   v=100
        elif data['earn_growth'] > 0.10: v=75
        elif data['earn_growth'] > 0:    v=40
        else:                             v=10
        grow += v; count += 1
        reasons.append(f"Earnings growth of {data['earn_growth']*100:.1f}% — {'accelerating' if v>=75 else 'modest' if v>=40 else 'declining'}")

    scores['growth'] = grow / count if count else 50

    # --- MOMENTUM SCORE (0-100) ---
    mom = 0
    count = 0
    if data['price'] and data['week52_high'] and data['week52_low']:
        range_pos = (data['price'] - data['week52_low']) / (data['week52_high'] - data['week52_low'])
        if range_pos > 0.75:   v=90
        elif range_pos > 0.50: v=65
        elif range_pos > 0.25: v=40
        else:                   v=20
        mom += v; count += 1
        reasons.append(f"Trading at {range_pos*100:.0f}% of 52-week range — {'near highs' if v>=90 else 'mid-range' if v>=65 else 'near lows'}")

    if data['price'] and data['ma50'] and data['ma200']:
        if data['price'] > data['ma50'] > data['ma200']:   v=100  # golden cross
        elif data['price'] > data['ma200']:                 v=65
        elif data['price'] > data['ma50']:                  v=50
        else:                                               v=20
        mom += v; count += 1
        reasons.append(f"Price vs MAs — {'bullish trend' if v>=100 else 'above 200MA' if v>=65 else 'weak trend'}")

    scores['momentum'] = mom / count if count else 50

    # --- COMPOSITE & SIGNAL ---
    composite = (
        scores['valuation'] * 0.30 +
        scores['quality']   * 0.30 +
        scores['growth']    * 0.20 +
        scores['momentum']  * 0.20
    )

    if composite >= 65:   signal, color = 'BUY',  '🟢'
    elif composite >= 45: signal, color = 'HOLD', '🟡'
    else:                 signal, color = 'SELL', '🔴'

    return scores, composite, signal, color, reasons

scores, composite, signal, color, reasons = generate_signal(data)
print(f"✅ Signal generated: {color} {signal} (score: {composite:.1f}/100)")

✅ Signal generated: 🟡 HOLD (score: 54.9/100)


In [5]:
def print_brief(ticker, data, scores, composite, signal, color, reasons):
    mktcap_b = f"${data['mktcap']/1e9:.1f}B" if data['mktcap'] else 'N/A'
    
    def fmt_pct(v): return f"{v*100:.1f}%" if v else 'N/A'
    def fmt_x(v):   return f"{v:.1f}x" if v else 'N/A'
    def fmt_n(v):   return f"{v:.1f}" if v else 'N/A'
    def fmt_fcf(v): return f"${v/1e9:.1f}B" if v else 'N/A'
    def fmt_div(v):
        if not v: return 'None'
        return f"{v*100:.1f}%" if v < 0.5 else f"{v:.1f}%"

    div = fmt_div(data['div_yield'])

    print(f"""
{'='*60}
  INVESTMENT BRIEF — {datetime.today().strftime('%B %d, %Y')}
{'='*60}
  {data['name']} ({ticker.upper()})
  {data['sector']} | {data['industry']}
  Price: ${data['price']:.2f}  |  Mkt Cap: {mktcap_b}  |  Beta: {fmt_n(data['beta'])}
  Dividend Yield: {div}
{'='*60}

  SIGNAL: {color} {signal}  ({composite:.1f} / 100)

{'─'*60}
  SCORES
{'─'*60}
  Valuation  : {scores['valuation']:.1f}/100  {'█' * int(scores['valuation']//10)}
  Quality    : {scores['quality']:.1f}/100  {'█' * int(scores['quality']//10)}
  Growth     : {scores['growth']:.1f}/100  {'█' * int(scores['growth']//10)}
  Momentum   : {scores['momentum']:.1f}/100  {'█' * int(scores['momentum']//10)}

{'─'*60}
  VALUATION
{'─'*60}
  Trailing P/E   : {fmt_x(data['pe'])}
  Forward P/E    : {fmt_x(data['fwd_pe'])}
  PEG Ratio      : {fmt_n(data['peg'])}
  EV/EBITDA      : {fmt_x(data['ev_ebitda'])}
  Price/Book     : {fmt_x(data['pb'])}

{'─'*60}
  QUALITY
{'─'*60}
  ROE            : {fmt_pct(data['roe'])}
  ROA            : {fmt_pct(data['roa'])}
  Gross Margin   : {fmt_pct(data['gross_margin'])}
  Operating Margin: {fmt_pct(data['op_margin'])}
  Net Margin     : {fmt_pct(data['net_margin'])}
  Debt/Equity    : {fmt_n(data['de_ratio'])}
  Current Ratio  : {fmt_n(data['current_r'])}
  Free Cash Flow : {fmt_fcf(data['fcf'])}

{'─'*60}
  GROWTH
{'─'*60}
  Revenue Growth : {fmt_pct(data['rev_growth'])}
  Earnings Growth: {fmt_pct(data['earn_growth'])}

{'─'*60}
  MOMENTUM
{'─'*60}
  52W High       : ${data['week52_high']:.2f}
  52W Low        : ${data['week52_low']:.2f}
  50D MA         : ${data['ma50']:.2f}
  200D MA        : ${data['ma200']:.2f}
  vs 52W High    : {((data['price']/data['week52_high'])-1)*100:.1f}%

{'─'*60}
  KEY OBSERVATIONS
{'─'*60}""")
    for r in reasons:
        print(f"  • {r}")
    print(f"""
{'─'*60}
  ⚠️  This is a quantitative signal only. Always do your
  own due diligence before making investment decisions.
{'='*60}
""")

# Run it
print_brief(ticker, data, scores, composite, signal, color, reasons)


  INVESTMENT BRIEF — June 14, 2026
  Mastercard Incorporated (MA)
  Financial Services | Credit Services
  Price: $489.98  |  Mkt Cap: $432.9B  |  Beta: 0.7
  Dividend Yield: 0.7%

  SIGNAL: 🟡 HOLD  (54.9 / 100)

────────────────────────────────────────────────────────────
  SCORES
────────────────────────────────────────────────────────────
  Valuation  : 52.5/100  █████
  Quality    : 58.8/100  █████
  Growth     : 87.5/100  ████████
  Momentum   : 20.0/100  ██

────────────────────────────────────────────────────────────
  VALUATION
────────────────────────────────────────────────────────────
  Trailing P/E   : 28.3x
  Forward P/E    : 21.5x
  PEG Ratio      : 1.5
  EV/EBITDA      : 20.8x
  Price/Book     : 64.7x

────────────────────────────────────────────────────────────
  QUALITY
────────────────────────────────────────────────────────────
  ROE            : 232.1%
  ROA            : 25.0%
  Gross Margin   : 100.0%
  Operating Margin: 60.8%
  Net Margin     : 45.9%
  Debt/Equit

In [6]:
def analyze(ticker):
    ticker = ticker.strip().upper()
    print(f"Fetching data for {ticker}...")
    try:
        data = get_fundamentals(ticker)
        scores, composite, signal, color, reasons = generate_signal(data)
        print_brief(ticker, data, scores, composite, signal, color, reasons)
    except Exception as e:
        print(f"❌ Could not fetch data for {ticker}. Check the ticker and try again. ({e})")

# ── CHANGE THIS TICKER ANYTIME ──
analyze('NVDA')

Fetching data for NVDA...

  INVESTMENT BRIEF — June 14, 2026
  NVIDIA Corporation (NVDA)
  Technology | Semiconductors
  Price: $205.19  |  Mkt Cap: $4969.9B  |  Beta: 2.2
  Dividend Yield: 49.0%

  SIGNAL: 🟢 BUY  (81.4 / 100)

────────────────────────────────────────────────────────────
  SCORES
────────────────────────────────────────────────────────────
  Valuation  : 61.2/100  ██████
  Quality    : 100.0/100  ██████████
  Growth     : 100.0/100  ██████████
  Momentum   : 65.0/100  ██████

────────────────────────────────────────────────────────────
  VALUATION
────────────────────────────────────────────────────────────
  Trailing P/E   : 31.4x
  Forward P/E    : 16.1x
  PEG Ratio      : 0.6
  EV/EBITDA      : 29.8x
  Price/Book     : 25.4x

────────────────────────────────────────────────────────────
  QUALITY
────────────────────────────────────────────────────────────
  ROE            : 114.3%
  ROA            : 52.7%
  Gross Margin   : 74.1%
  Operating Margin: 65.6%
  Net Mar

In [7]:
analyze('SPCX')

Fetching data for SPCX...

  INVESTMENT BRIEF — June 14, 2026
  Space Exploration Technologies Corp. (SPCX)
  Industrials | Aerospace & Defense
  Price: $160.95  |  Mkt Cap: $2107.0B  |  Beta: N/A
  Dividend Yield: None

  SIGNAL: 🟡 HOLD  (52.5 / 100)

────────────────────────────────────────────────────────────
  SCORES
────────────────────────────────────────────────────────────
  Valuation  : 55.0/100  █████
  Quality    : 50.0/100  █████
  Growth     : 75.0/100  ███████
  Momentum   : 30.0/100  ███

────────────────────────────────────────────────────────────
  VALUATION
────────────────────────────────────────────────────────────
  Trailing P/E   : N/A
  Forward P/E    : -1788.3x
  PEG Ratio      : N/A
  EV/EBITDA      : 239.8x
  Price/Book     : 27.0x

────────────────────────────────────────────────────────────
  QUALITY
────────────────────────────────────────────────────────────
  ROE            : N/A
  ROA            : N/A
  Gross Margin   : 48.8%
  Operating Margin: -41.6%
 

In [8]:
# Fetch ABT data using your existing tool
ticker = 'ABT'
data = get_fundamentals(ticker)
scores, composite, signal, color, reasons = generate_signal(data)
print_brief(ticker, data, scores, composite, signal, color, reasons)


  INVESTMENT BRIEF — June 14, 2026
  Abbott Laboratories (ABT)
  Healthcare | Medical Devices
  Price: $88.18  |  Mkt Cap: $153.6B  |  Beta: 0.6
  Dividend Yield: 2.9%

  SIGNAL: 🟡 HOLD  (49.0 / 100)

────────────────────────────────────────────────────────────
  SCORES
────────────────────────────────────────────────────────────
  Valuation  : 75.0/100  ███████
  Quality    : 55.0/100  █████
  Growth     : 30.0/100  ███
  Momentum   : 20.0/100  ██

────────────────────────────────────────────────────────────
  VALUATION
────────────────────────────────────────────────────────────
  Trailing P/E   : 24.7x
  Forward P/E    : 14.6x
  PEG Ratio      : 1.3
  EV/EBITDA      : 15.4x
  Price/Book     : 3.0x

────────────────────────────────────────────────────────────
  QUALITY
────────────────────────────────────────────────────────────
  ROE            : 12.3%
  ROA            : 5.6%
  Gross Margin   : 56.5%
  Operating Margin: 13.5%
  Net Margin     : 13.9%
  Debt/Equity    : 64.8
  Curre

In [9]:
memo = """
================================================================
  INVESTMENT MEMO
  Abbott Laboratories (ABT) — LONG
  Date: June 14, 2026
  Prepared by: Yashvi Shah, CFA Level 3
================================================================

BUSINESS OVERVIEW
─────────────────
Abbott Laboratories is a global diversified healthcare company
operating across four segments:

  • Medical Devices (40% of revenue) — market leader in
    continuous glucose monitors (CGM) via FreeStyle Libre,
    structural heart, and electrophysiology

  • Diagnostics (25%) — rapid testing and core lab diagnostics;
    benefited significantly from COVID-19 testing (2021-22)

  • Established Pharmaceuticals (20%) — branded generics across
    emerging markets

  • Nutrition (15%) — Ensure, Similac, and pediatric nutrition

Market Cap : $153.6B
Price      : $88.18
52W Range  : $81.97 – $139.06
Dividend   : 2.9% yield (52 consecutive years of increases)

================================================================
INVESTMENT THESIS — 3 REASONS TO BUY
================================================================

1. POST-COVID NORMALIZATION IS NEARLY COMPLETE
───────────────────────────────────────────────
ABT's earnings declined -19.7% YoY — but this is misleading.
The decline is almost entirely driven by the wind-down of
COVID-19 testing revenue ($7.7B at peak in 2022, now <$500M).
Strip out diagnostics and the core business — Medical Devices
and Nutrition — is growing at high single digits.

FreeStyle Libre (CGM) alone grew ~20% YoY and is on track for
$7B+ in annual revenue. As COVID noise fully exits the base,
reported earnings growth will re-accelerate in 2025-26.

Key metric: Forward P/E of 14.6x prices in continued weakness
— not the recovery that is already underway.

2. FREESTYLE LIBRE IS A STRUCTURAL GROWTH DRIVER
──────────────────────────────────────────────────
The global CGM market is projected to grow at ~20% CAGR
through 2030, driven by:
  • 537 million diabetics globally, <10% currently using CGM
  • Expansion into non-diabetic metabolic health monitoring
  • Reimbursement expansion across Europe and emerging markets

FreeStyle Libre holds ~50% global CGM market share and is
the #1 CGM device in 60+ countries. This is a durable,
recurring-revenue business with high switching costs —
exactly the kind of asset the market is undervaluing while
focused on the COVID testing decline.

3. ATTRACTIVE VALUATION WITH MARGIN OF SAFETY
───────────────────────────────────────────────
  Metric          ABT       Sector Avg   vs Peers
  ─────────────────────────────────────────────
  Forward P/E     14.6x     22.0x        -34% discount
  EV/EBITDA       15.4x     18.0x        -14% discount
  PEG Ratio       1.3x      1.8x         -28% discount
  Dividend Yield  2.9%      1.4%         2x sector avg

At $88, ABT trades at a 34% discount to the sector on
forward earnings — despite being a higher-quality business
than most peers. The stock has not been this cheap relative
to the market since 2013.

================================================================
VALUATION — PRICE TARGET
================================================================

Base case: ABT returns to 20x forward earnings as COVID
noise exits and FreeStyle Libre growth re-accelerates.

  Forward EPS estimate (2026)  : ~$5.10
  Target multiple              : 20x (sector avg)
  Price target                 : $102
  Upside from current price    : +15.7%
  + Dividend yield             : +2.9%
  ─────────────────────────────────────
  Total expected return        : ~18-19% over 12 months

Bull case ($118): Libre accelerates to $8B revenue,
multiple re-rates to 23x. Upside: +34%

Bear case ($78): Macro headwinds, Libre competition
intensifies. Downside: -12%

Risk/Reward: ~2.8x favorable

================================================================
RISKS
================================================================

  1. CGM Competition — Dexcom and emerging players could
     pressure Libre's market share and pricing power

  2. Emerging Market Exposure — ~20% of revenue from
     developing markets; FX and political risk is real

  3. Litigation — ongoing talc-related legal liability
     from legacy businesses; hard to quantify

  4. Earnings Visibility — until COVID testing fully
     rolls off, reported numbers will remain noisy and
     may spook short-term investors

================================================================
TECHNICAL SETUP
================================================================

  Current price  : $88.18
  52W Low        : $81.97  ← strong support ~$82
  50D MA         : $91.22  ← first resistance
  200D MA        : $114.49 ← longer-term target zone

  The stock is trading 11% above its 52-week low with
  strong support at $82. A stop-loss at $80 (-9% from
  current) gives a clean entry with defined downside.

================================================================
SIGNAL FROM QUANTITATIVE SCREENER
================================================================

  Composite Score : 49.0 / 100  (HOLD — edge of BUY)
  Valuation Score : 75.0 / 100  ✅ Strong
  Quality Score   : 55.0 / 100  ✅ Adequate
  Growth Score    : 30.0 / 100  ⚠️  Depressed (COVID base)
  Momentum Score  : 20.0 / 100  ⚠️  Oversold

  Note: The low growth and momentum scores reflect the
  COVID earnings comparison effect and recent selloff —
  both of which are mean-reverting, not structural. As
  these normalize, the composite score should move into
  BUY territory, making this a forward-looking opportunity
  the quant model is not yet fully capturing.

================================================================
CONCLUSION
================================================================

Abbott is a high-quality, diversified healthcare compounder
trading at a 34% discount to sector peers due to temporary
COVID base effects and broad healthcare sector weakness.

FreeStyle Libre is a structural growth business in a market
growing at 20% CAGR. The dividend has been raised for 52
consecutive years. Free cash flow of $6.3B funds both the
dividend and continued R&D investment.

At $88 with a $102 price target and $82 support, the
risk/reward is approximately 2.8x favorable.

  RECOMMENDATION : LONG ABT
  ENTRY           : $88 (current)
  PRICE TARGET    : $102 (base) / $118 (bull)
  STOP LOSS       : $80
  TIME HORIZON    : 12 months
  POSITION SIZE   : 3-5% of portfolio

================================================================
DISCLAIMER: This memo reflects personal investment research
and is not financial advice. Always conduct independent
due diligence before making investment decisions.
================================================================
"""

print(memo)

# Save it as a markdown file
with open('ABT_investment_memo.md', 'w') as f:
    f.write(memo)

print("✅ Saved as ABT_investment_memo.md")


  INVESTMENT MEMO
  Abbott Laboratories (ABT) — LONG
  Date: June 14, 2026
  Prepared by: Yashvi Shah, CFA Level 3

BUSINESS OVERVIEW
─────────────────
Abbott Laboratories is a global diversified healthcare company
operating across four segments:

  • Medical Devices (40% of revenue) — market leader in
    continuous glucose monitors (CGM) via FreeStyle Libre,
    structural heart, and electrophysiology

  • Diagnostics (25%) — rapid testing and core lab diagnostics;
    benefited significantly from COVID-19 testing (2021-22)

  • Established Pharmaceuticals (20%) — branded generics across
    emerging markets

  • Nutrition (15%) — Ensure, Similac, and pediatric nutrition

Market Cap : $153.6B
Price      : $88.18
52W Range  : $81.97 – $139.06
Dividend   : 2.9% yield (52 consecutive years of increases)

INVESTMENT THESIS — 3 REASONS TO BUY

1. POST-COVID NORMALIZATION IS NEARLY COMPLETE
───────────────────────────────────────────────
ABT's earnings declined -19.7% YoY — but this is mi

In [10]:
def generate_memo(ticker, entry_price=None, target_multiple=20, stop_pct=0.09, horizon="12 months", position_size="3-5%"):
    
    # Fetch data
    print(f"Fetching data for {ticker}...")
    ticker = ticker.strip().upper()
    data = get_fundamentals(ticker)
    scores, composite, signal, color, reasons = generate_signal(data)
    
    # Auto-fill entry price
    price = data['price']
    entry = entry_price or price
    
    # Price target calculation
    fwd_pe = data['fwd_pe'] or data['pe'] or 20
    fwd_eps = price / fwd_pe
    target = round(fwd_eps * target_multiple, 2)
    upside = round((target / entry - 1) * 100, 1)
    stop = round(entry * (1 - stop_pct), 2)
    div = data['div_yield'] * 100 if data['div_yield'] and data['div_yield'] < 0.5 else (data['div_yield'] or 0)
    total_return = round(upside + div, 1)
    
    # Risk/reward
    downside = round(stop_pct * 100, 1)
    rr = round(upside / downside, 1) if downside else 'N/A'
    
    # Formatters
    def fmt_pct(v): return f"{v*100:.1f}%" if v else 'N/A'
    def fmt_x(v):   return f"{v:.1f}x" if v else 'N/A'
    def fmt_n(v):   return f"{v:.1f}" if v else 'N/A'
    def fmt_fcf(v): return f"${v/1e9:.1f}B" if v else 'N/A'
    def fmt_div(v):
        if not v: return 'None'
        return f"{v*100:.1f}%" if v < 0.5 else f"{v:.1f}%"
    
    mktcap = f"${data['mktcap']/1e9:.1f}B" if data['mktcap'] else 'N/A'
    sig_bar = '█' * int(composite // 10)

    memo = f"""
================================================================
  INVESTMENT MEMO
  {data['name']} ({ticker}) — LONG
  Date: {datetime.today().strftime('%B %d, %Y')}
  Prepared by: Yashvi Shah, CFA Level 3
================================================================

COMPANY OVERVIEW
────────────────
  Name      : {data['name']}
  Sector    : {data['sector']}
  Industry  : {data['industry']}
  Market Cap: {mktcap}
  Beta      : {fmt_n(data['beta'])}
  Dividend  : {fmt_div(data['div_yield'])}

================================================================
QUANTITATIVE SIGNAL (from Daily Screener)
================================================================

  Composite Score : {composite:.1f} / 100  {sig_bar}
  Signal          : {color} {signal}

  Valuation Score : {scores['valuation']:.1f}/100  {'█' * int(scores['valuation']//10)}
  Quality Score   : {scores['quality']:.1f}/100  {'█' * int(scores['quality']//10)}
  Growth Score    : {scores['growth']:.1f}/100  {'█' * int(scores['growth']//10)}
  Momentum Score  : {scores['momentum']:.1f}/100  {'█' * int(scores['momentum']//10)}

================================================================
VALUATION SNAPSHOT
================================================================

  Trailing P/E    : {fmt_x(data['pe'])}
  Forward P/E     : {fmt_x(data['fwd_pe'])}
  PEG Ratio       : {fmt_n(data['peg'])}
  EV/EBITDA       : {fmt_x(data['ev_ebitda'])}
  Price/Book      : {fmt_x(data['pb'])}

================================================================
QUALITY SNAPSHOT
================================================================

  ROE             : {fmt_pct(data['roe'])}
  ROA             : {fmt_pct(data['roa'])}
  Gross Margin    : {fmt_pct(data['gross_margin'])}
  Operating Margin: {fmt_pct(data['op_margin'])}
  Net Margin      : {fmt_pct(data['net_margin'])}
  Debt/Equity     : {fmt_n(data['de_ratio'])}
  Current Ratio   : {fmt_n(data['current_r'])}
  Free Cash Flow  : {fmt_fcf(data['fcf'])}

================================================================
GROWTH SNAPSHOT
================================================================

  Revenue Growth  : {fmt_pct(data['rev_growth'])}
  Earnings Growth : {fmt_pct(data['earn_growth'])}

================================================================
TECHNICAL SETUP
================================================================

  Current Price   : ${price:.2f}
  52W High        : ${data['week52_high']:.2f}
  52W Low         : ${data['week52_low']:.2f}
  vs 52W High     : {((price/data['week52_high'])-1)*100:.1f}%
  50D MA          : ${data['ma50']:.2f}
  200D MA         : ${data['ma200']:.2f}

================================================================
KEY OBSERVATIONS
================================================================
"""
    for r in reasons:
        memo += f"  • {r}\n"

    memo += f"""
================================================================
TRADE SETUP
================================================================

  Entry Price     : ${entry:.2f}
  Price Target    : ${target:.2f}  (based on {target_multiple}x fwd earnings)
  Stop Loss       : ${stop:.2f}  (-{stop_pct*100:.0f}% from entry)
  Upside          : +{upside}%
  Dividend Yield  : +{div:.1f}%
  Total Return    : ~{total_return}% over {horizon}
  Risk/Reward     : {rr}x favorable
  Position Size   : {position_size} of portfolio
  Time Horizon    : {horizon}

================================================================
DISCLAIMER: This memo reflects personal investment research
and is not financial advice. Always conduct independent
due diligence before making investment decisions.
================================================================
"""

    print(memo)
    
    # Save as markdown file
    filename = f"{ticker}_investment_memo.md"
    with open(filename, 'w') as f:
        f.write(memo)
    print(f"✅ Saved as {filename}")

# ── USE IT LIKE THIS ──────────────────────────────────────
generate_memo('ABT')

Fetching data for ABT...

  INVESTMENT MEMO
  Abbott Laboratories (ABT) — LONG
  Date: June 14, 2026
  Prepared by: Yashvi Shah, CFA Level 3

COMPANY OVERVIEW
────────────────
  Name      : Abbott Laboratories
  Sector    : Healthcare
  Industry  : Medical Devices
  Market Cap: $153.6B
  Beta      : 0.6
  Dividend  : 2.9%

QUANTITATIVE SIGNAL (from Daily Screener)

  Composite Score : 49.0 / 100  ████
  Signal          : 🟡 HOLD

  Valuation Score : 75.0/100  ███████
  Quality Score   : 55.0/100  █████
  Growth Score    : 30.0/100  ███
  Momentum Score  : 20.0/100  ██

VALUATION SNAPSHOT

  Trailing P/E    : 24.7x
  Forward P/E     : 14.6x
  PEG Ratio       : 1.3
  EV/EBITDA       : 15.4x
  Price/Book      : 3.0x

QUALITY SNAPSHOT

  ROE             : 12.3%
  ROA             : 5.6%
  Gross Margin    : 56.5%
  Operating Margin: 13.5%
  Net Margin      : 13.9%
  Debt/Equity     : 64.8
  Current Ratio   : 1.4
  Free Cash Flow  : $6.3B

GROWTH SNAPSHOT

  Revenue Growth  : 7.8%
  Earnings G

In [11]:
# Default — just a ticker
generate_memo('NVDA')

# With custom assumptions
generate_memo('AAPL', target_multiple=25, stop_pct=0.08, position_size="5%")

Fetching data for NVDA...

  INVESTMENT MEMO
  NVIDIA Corporation (NVDA) — LONG
  Date: June 14, 2026
  Prepared by: Yashvi Shah, CFA Level 3

COMPANY OVERVIEW
────────────────
  Name      : NVIDIA Corporation
  Sector    : Technology
  Industry  : Semiconductors
  Market Cap: $4969.9B
  Beta      : 2.2
  Dividend  : 49.0%

QUANTITATIVE SIGNAL (from Daily Screener)

  Composite Score : 81.4 / 100  ████████
  Signal          : 🟢 BUY

  Valuation Score : 61.2/100  ██████
  Quality Score   : 100.0/100  ██████████
  Growth Score    : 100.0/100  ██████████
  Momentum Score  : 65.0/100  ██████

VALUATION SNAPSHOT

  Trailing P/E    : 31.4x
  Forward P/E     : 16.1x
  PEG Ratio       : 0.6
  EV/EBITDA       : 29.8x
  Price/Book      : 25.4x

QUALITY SNAPSHOT

  ROE             : 114.3%
  ROA             : 52.7%
  Gross Margin    : 74.1%
  Operating Margin: 65.6%
  Net Margin      : 63.0%
  Debt/Equity     : 6.6
  Current Ratio   : 3.4
  Free Cash Flow  : $46.3B

GROWTH SNAPSHOT

  Revenue Gro